In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
with open(r"C:\Projects\Transformers\Combined.txt","r",encoding="utf-8") as T:
    T = T.read(500000)

chars = sorted(set(T.split()))
vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}  
itos = {i: c for i, c in enumerate(chars)} 
def encode(s):
    return [stoi[c] for c in s.split()]
def decode(indices):
    return ' '.join([itos[i] for i in indices])

text = encode(T)

87998


In [14]:

seq_len = 8
batch = 8
def get_batch():
    max_start = len(text) - seq_len - 1
    ix = torch.randint(0,max_start , (batch,))

    x = torch.stack([
        torch.tensor(text[i:i+seq_len], dtype=torch.long)
        for i in ix
    ])
    
    y = torch.stack([
        torch.tensor(text[i+1:i+seq_len+1], dtype=torch.long)
        for i in ix
    ])


    x = x
    y = y
    return x, y

class Bigram(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, 512)
        self.fc1 = nn.Linear(512, 512)
        self.fc2 = nn.Linear(512, vocab)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.emb(x)      
        x = self.fc1(x)     
        x = self.act(x)
        x = self.fc2(x)     
        return x


model = Bigram(vocab)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4)
loss_type = nn.CrossEntropyLoss()

In [15]:
for i in range(8000):
    x, y = get_batch()

    logits = model(x)            

    B, T, V = logits.shape
    logits = logits.view(B*T, V)
    y = y.view(B*T)

    loss = loss_type(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Step {i}, Loss: {loss.item()}")


Step 0, Loss: 9.669329643249512
Step 100, Loss: 8.478830337524414
Step 200, Loss: 7.742800235748291
Step 300, Loss: 7.585514068603516
Step 400, Loss: 6.978274345397949
Step 500, Loss: 6.3386945724487305
Step 600, Loss: 6.743587493896484
Step 700, Loss: 6.238465309143066
Step 800, Loss: 6.539920330047607
Step 900, Loss: 6.785300254821777
Step 1000, Loss: 7.028192520141602
Step 1100, Loss: 6.417860984802246
Step 1200, Loss: 6.440706729888916
Step 1300, Loss: 7.003811836242676
Step 1400, Loss: 6.5012922286987305
Step 1500, Loss: 6.135969161987305
Step 1600, Loss: 7.08087682723999
Step 1700, Loss: 6.694048881530762
Step 1800, Loss: 6.58955192565918
Step 1900, Loss: 6.071104526519775
Step 2000, Loss: 6.060478210449219


KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def generate(model, start_token, max_new_tokens=25, temperature=1.0):
    model.eval()

    idx = torch.tensor([start_token], dtype=torch.long)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -seq_len:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
         
        logits = logits / temperature

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_token], dim=1)

    return idx

start_char = 'To the'
start_token = encode(start_char) 

out = generate(model, start_token, max_new_tokens=300, temperature=0.7)
print(decode(out[0].tolist()))


To the thithe acke d lis o theng an ge he r, m, d s sherte t t to toud he s athagey lear apererurldescatherlllis hes ayes t as e t tot wher hes ofind s s tom, ce sted bout her wimisth st otis f hay hay neshed a wno f theer sis ld sabo te s wnsas sher busth allle fe w s out. bate t ou we ker s wo oret im w
